In [2]:
# Mount Drive and load the four tables needed for KPI 13
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

po_h = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
suppliers = pd.read_csv(BASE_PATH + 'suppliers.csv')
products = pd.read_csv(BASE_PATH + 'products.csv')

tables = {'purchase orders header (po_h)': po_h,
          'purchase orders lines (po_l)': po_l,
          'suppliers': suppliers,
          'products': products}

for name, t in tables.items():
    print('=' * 60)
    print(name, '| rows:', t.shape[0], '| columns:', t.shape[1])
    print(list(t.columns))
    print(t.head(3).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
purchase orders header (po_h) | rows: 24000 | columns: 10
['po_id', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total']
       po_id supplier_id branch_id  order_date expected_delivery_date received_date po_status   total_cost  total_gst_amount   grand_total
0  PO-141186     SUP0001    PUN001  2019-04-10             2019-04-29    2019-04-29  Received  22955232.11      6.427465e+06  2.938270e+07
1  PO-215234     SUP0005    DEL001  2020-12-30             2021-01-27    2021-01-27  Received  19406544.39      5.428358e+06  2.483490e+07
2  PO-296826     SUP0008    AHM001  2022-04-05             2022-05-01    2022-05-01  Received   1295425.79      3.627192e+05  1.658145e+06
purchase orders lines (po_l) | rows: 155495 | columns: 9
['po_id', 'line_number', 'product_id', 'qua

In [3]:
# Step 2: order status counts, and promised vs real delivery date for Received orders
print('Orders by status:')
print(po_h['po_status'].value_counts(dropna=False))

rec = po_h[po_h['po_status'] == 'Received'].copy()
rec['expected'] = pd.to_datetime(rec['expected_delivery_date'])
rec['received'] = pd.to_datetime(rec['received_date'])
rec['days_late'] = (rec['received'] - rec['expected']).dt.days

print()
print('Received orders:', len(rec))
print('Missing dates in Received orders:')
print(rec[['order_date', 'expected_delivery_date', 'received_date']].isnull().sum())
print()
print('Late (received after promised):', (rec['days_late'] > 0).sum())
print('On the promised day:', (rec['days_late'] == 0).sum())
print('Early:', (rec['days_late'] < 0).sum())

Orders by status:
po_status
Received     21630
Cancelled     2370
Name: count, dtype: int64

Received orders: 21630
Missing dates in Received orders:
order_date                0
expected_delivery_date    0
received_date             0
dtype: int64

Late (received after promised): 3267
On the promised day: 17392
Early: 971


In [4]:
# Step 3: how many days late or early the Received orders are
print('Smallest days late (negative = early):', rec['days_late'].min())
print('Largest days late:', rec['days_late'].max())
print('Average days late:', round(rec['days_late'].mean(), 3))
print()
print('Orders for each number of days late (negative = early):')
print(rec['days_late'].value_counts().sort_index())

Smallest days late (negative = early): -3
Largest days late: 7
Average days late: 0.513

Orders for each number of days late (negative = early):
days_late
-3      334
-2      300
-1      337
 0    17392
 1      467
 2      475
 3      461
 4      469
 5      464
 6      484
 7      447
Name: count, dtype: int64


In [5]:
# Step 4: lateness and real lead time for each supplier
rec['real_lead_days'] = (rec['received'] - pd.to_datetime(rec['order_date'])).dt.days

by_sup = rec.groupby('supplier_id').agg(
    orders=('po_id', 'count'),
    late_orders=('days_late', lambda s: (s > 0).sum()),
    average_days_late=('days_late', 'mean'),
    average_real_lead_days=('real_lead_days', 'mean')
).reset_index()

by_sup['late_share_percent'] = (by_sup['late_orders'] / by_sup['orders'] * 100).round(2)
by_sup['average_days_late'] = by_sup['average_days_late'].round(3)
by_sup['average_real_lead_days'] = by_sup['average_real_lead_days'].round(2)
by_sup = by_sup.merge(suppliers[['supplier_id', 'supplier_name']], on='supplier_id', how='left')

print(by_sup[['supplier_id', 'supplier_name', 'orders', 'late_orders',
              'late_share_percent', 'average_days_late',
              'average_real_lead_days']].to_string(index=False))
print()
print('Total orders in this table:', by_sup['orders'].sum())

supplier_id                   supplier_name  orders  late_orders  late_share_percent  average_days_late  average_real_lead_days
    SUP0001           Shenzhen OEM Supplies    2653          400               15.08              0.496                   19.50
    SUP0002 Guangzhou Local Vendor Supplies    2635          396               15.03              0.516                   10.52
    SUP0003   Shanghai Distributor Supplies    2723          408               14.98              0.531                   16.53
    SUP0004     Ningbo Distributor Supplies    2664          404               15.17              0.497                   18.50
    SUP0005            Tianjin OEM Supplies    2754          404               14.67              0.485                   28.49
    SUP0006    Qingdao Distributor Supplies    2752          407               14.79              0.501                   10.50
    SUP0007    Suzhou Local Vendor Supplies    2727          418               15.33              0.528 

In [6]:
# Step 5: is the difference in late share between suppliers more than chance
from scipy.stats import chi2_contingency

by_sup['on_time_or_early_orders'] = by_sup['orders'] - by_sup['late_orders']
table = by_sup[['late_orders', 'on_time_or_early_orders']].values
chi2, p_value, dof, expected = chi2_contingency(table)

print('Chance of differences this big if all 8 suppliers were equally late (p-value):', round(p_value, 4))
print()

check = by_sup[['supplier_name', 'late_orders']].copy()
check['late_orders_if_all_equal'] = expected[:, 0].round(1)
check['difference'] = (check['late_orders'] - check['late_orders_if_all_equal']).round(1)
print(check.to_string(index=False))

Chance of differences this big if all 8 suppliers were equally late (p-value): 0.9702

                  supplier_name  late_orders  late_orders_if_all_equal  difference
          Shenzhen OEM Supplies          400                     400.7        -0.7
Guangzhou Local Vendor Supplies          396                     398.0        -2.0
  Shanghai Distributor Supplies          408                     411.3        -3.3
    Ningbo Distributor Supplies          404                     402.4         1.6
           Tianjin OEM Supplies          404                     416.0       -12.0
   Qingdao Distributor Supplies          407                     415.7        -8.7
   Suzhou Local Vendor Supplies          418                     411.9         6.1
          Hangzhou OEM Supplies          430                     411.1        18.9


In [7]:
# Show the supplier table numbers next to what really happened
compare = by_sup.merge(suppliers[['supplier_id', 'lead_time_days', 'reliability_score']],
                       on='supplier_id', how='left')
compare['real_minus_promised_days'] = (compare['average_real_lead_days'] - compare['lead_time_days']).round(2)

print(compare[['supplier_name', 'lead_time_days', 'average_real_lead_days',
               'real_minus_promised_days', 'reliability_score',
               'late_share_percent']].to_string(index=False))

                  supplier_name  lead_time_days  average_real_lead_days  real_minus_promised_days  reliability_score  late_share_percent
          Shenzhen OEM Supplies              19                   19.50                      0.50                  5               15.08
Guangzhou Local Vendor Supplies              10                   10.52                      0.52                  5               15.03
  Shanghai Distributor Supplies              16                   16.53                      0.53                  4               14.98
    Ningbo Distributor Supplies              18                   18.50                      0.50                  3               15.17
           Tianjin OEM Supplies              28                   28.49                      0.49                  3               14.67
   Qingdao Distributor Supplies              10                   10.50                      0.50                  3               14.79
   Suzhou Local Vendor Supplies          

In [8]:
# Count missing values in the columns KPI 13 uses and check every order's supplier is in the suppliers table
cols = ['po_id', 'supplier_id', 'order_date', 'expected_delivery_date', 'received_date']
print('Missing values in Received orders:')
print(rec[cols].isnull().sum())
print()
print('Orders whose supplier is not in the suppliers table:',
      (~rec['supplier_id'].isin(suppliers['supplier_id'])).sum())

Missing values in Received orders:
po_id                     0
supplier_id               0
order_date                0
expected_delivery_date    0
received_date             0
dtype: int64

Orders whose supplier is not in the suppliers table: 0


In [9]:
# Look for impossible dates: received or promised on or before the order date
rec['order'] = pd.to_datetime(rec['order_date'])

print('Received on or before the order date:', (rec['received'] <= rec['order']).sum())
print('Promised on or before the order date:', (rec['expected'] <= rec['order']).sum())
print('Smallest real wait (days):', rec['real_lead_days'].min())
print('Smallest promised wait (days):', (rec['expected'] - rec['order']).dt.days.min())

Received on or before the order date: 0
Promised on or before the order date: 0
Smallest real wait (days): 7
Smallest promised wait (days): 10


In [10]:
# Check that each order appears only once
print('Rows in the Received orders table:', len(rec))
print('Different order IDs in it:', rec['po_id'].nunique())
print('Order IDs that appear more than once in it:', rec['po_id'].duplicated().sum())
print('Order IDs that appear more than once in the full orders table (24,000 rows):',
      po_h['po_id'].duplicated().sum())

Rows in the Received orders table: 21630
Different order IDs in it: 21630
Order IDs that appear more than once in it: 0
Order IDs that appear more than once in the full orders table (24,000 rows): 0


In [11]:
# Check that the per-supplier numbers add up to the totals for all Received orders
early = (rec['days_late'] < 0).sum()
on_day = (rec['days_late'] == 0).sum()
late = (rec['days_late'] > 0).sum()

print('Orders added up over the 8 suppliers:', by_sup['orders'].sum(), '| all Received orders:', len(rec))
print('Late orders added up over the 8 suppliers:', by_sup['late_orders'].sum(), '| all late Received orders:', late)
print('Early + on the promised day + late:', early + on_day + late)
print('Received + Cancelled orders:', len(rec) + (po_h['po_status'] == 'Cancelled').sum(), '| all orders:', len(po_h))

Orders added up over the 8 suppliers: 21630 | all Received orders: 21630
Late orders added up over the 8 suppliers: 3267 | all late Received orders: 3267
Early + on the promised day + late: 21630
Received + Cancelled orders: 24000 | all orders: 24000


In [12]:
# Recompute each supplier's three numbers a second way and compare with the supplier table
rows = []
for sid in sorted(rec['supplier_id'].unique()):
    part = rec[rec['supplier_id'] == sid]
    rows.append({'supplier_id': sid,
                 'recomputed_late_share_percent': round((part['received'] > part['expected']).mean() * 100, 2),
                 'recomputed_average_days_late': round((part['received'] - part['expected']).dt.days.mean(), 3),
                 'recomputed_average_real_lead_days': round((part['received'] - part['order']).dt.days.mean(), 2)})

second = pd.DataFrame(rows).merge(by_sup, on='supplier_id')

print('Largest difference, late share (percent):',
      (second['recomputed_late_share_percent'] - second['late_share_percent']).abs().max())
print('Largest difference, average days late:',
      (second['recomputed_average_days_late'] - second['average_days_late']).abs().max())
print('Largest difference, average real lead days:',
      (second['recomputed_average_real_lead_days'] - second['average_real_lead_days']).abs().max())

Largest difference, late share (percent): 0.0
Largest difference, average days late: 0.0
Largest difference, average real lead days: 0.0


In [13]:
# Look for extreme orders: smallest and largest days late, median against average real wait, longest real wait
out = rec.groupby('supplier_id').agg(
    smallest_days_late=('days_late', 'min'),
    largest_days_late=('days_late', 'max'),
    median_real_lead_days=('real_lead_days', 'median'),
    average_real_lead_days=('real_lead_days', 'mean'),
    longest_real_lead_days=('real_lead_days', 'max')).reset_index()

out['longest_vs_average'] = (out['longest_real_lead_days'] / out['average_real_lead_days']).round(2)
out = out.merge(suppliers[['supplier_id', 'supplier_name']], on='supplier_id')

print(out[['supplier_name', 'smallest_days_late', 'largest_days_late',
           'median_real_lead_days', 'average_real_lead_days',
           'longest_real_lead_days', 'longest_vs_average']].round(2).to_string(index=False))

                  supplier_name  smallest_days_late  largest_days_late  median_real_lead_days  average_real_lead_days  longest_real_lead_days  longest_vs_average
          Shenzhen OEM Supplies                  -3                  7                   19.0                   19.50                      26                1.33
Guangzhou Local Vendor Supplies                  -3                  7                   10.0                   10.52                      17                1.62
  Shanghai Distributor Supplies                  -3                  7                   16.0                   16.53                      23                1.39
    Ningbo Distributor Supplies                  -3                  7                   18.0                   18.50                      25                1.35
           Tianjin OEM Supplies                  -3                  7                   28.0                   28.49                      35                1.23
   Qingdao Distributor Suppl

In [14]:
# Count the months with orders for each supplier and show its first and last order month
rec['order_month'] = rec['order'].dt.to_period('M')

hist = rec.groupby('supplier_id').agg(
    months_with_orders=('order_month', 'nunique'),
    first_order_month=('order_month', 'min'),
    last_order_month=('order_month', 'max'))
hist['fewest_orders_in_a_month'] = rec.groupby(['supplier_id', 'order_month']).size().groupby('supplier_id').min()

print(hist.to_string())
print()
print('Months with orders, all suppliers together:', rec['order_month'].nunique())

             months_with_orders first_order_month last_order_month  fewest_orders_in_a_month
supplier_id                                                                                 
SUP0001                      72           2019-01          2024-12                        26
SUP0002                      72           2019-01          2024-12                        23
SUP0003                      72           2019-01          2024-12                        25
SUP0004                      72           2019-01          2024-12                        24
SUP0005                      72           2019-01          2024-12                        25
SUP0006                      72           2019-01          2024-12                        19
SUP0007                      72           2019-01          2024-12                        23
SUP0008                      72           2019-01          2024-12                        25

Months with orders, all suppliers together: 72


In [15]:
# Recalculate three suppliers from the raw orders file with a plain loop and compare with the supplier table
import csv
from datetime import date

picked = {'SUP0006': 'fastest wait', 'SUP0005': 'slowest wait', 'SUP0008': 'highest late share'}

with open(BASE_PATH + 'purchase_orders_header.csv', newline='') as f:
    raw = list(csv.DictReader(f))

for sid, why in picked.items():
    orders = late = days_late_sum = lead_sum = 0
    for r in raw:
        if r['supplier_id'] == sid and r['po_status'] == 'Received':
            o = date.fromisoformat(r['order_date'])
            e = date.fromisoformat(r['expected_delivery_date'])
            g = date.fromisoformat(r['received_date'])
            orders += 1
            late += g > e
            days_late_sum += (g - e).days
            lead_sum += (g - o).days
    t = by_sup[by_sup['supplier_id'] == sid].iloc[0]
    print(sid, '(' + why + ')')
    print('  from the raw file: orders', orders, '| late', late,
          '| late share', round(late / orders * 100, 2),
          '| average days late', round(days_late_sum / orders, 3),
          '| average real lead days', round(lead_sum / orders, 2))
    print('  supplier table:    orders', t['orders'], '| late', t['late_orders'],
          '| late share', t['late_share_percent'],
          '| average days late', t['average_days_late'],
          '| average real lead days', t['average_real_lead_days'])

SUP0006 (fastest wait)
  from the raw file: orders 2752 | late 407 | late share 14.79 | average days late 0.501 | average real lead days 10.5
  supplier table:    orders 2752 | late 407 | late share 14.79 | average days late 0.501 | average real lead days 10.5
SUP0005 (slowest wait)
  from the raw file: orders 2754 | late 404 | late share 14.67 | average days late 0.485 | average real lead days 28.49
  supplier table:    orders 2754 | late 404 | late share 14.67 | average days late 0.485 | average real lead days 28.49
SUP0008 (highest late share)
  from the raw file: orders 2722 | late 430 | late share 15.8 | average days late 0.546 | average real lead days 26.55
  supplier table:    orders 2722 | late 430 | late share 15.8 | average days late 0.546 | average real lead days 26.55
